# STA2112 Homework 1 (v2)

**Name:** Alec Neme  
**Student number:** 1084575857

## Q1: Prediction targets and evaluation

### Q1(a)

**(i)** The first priority should be removing patient ID. Patient ID does not help us predict for new patients, since a new patient's ID was not seen in the training data. Moreover, the hospital does not assign IDs based on how likely patients are to be readmitted; they are probably sequential numbers assigned as new patients arrive. Furthermore, if a patient is frequently readmitted, patients with similar IDs may be given a higher predicted likelihood of readmission, even though this is not a real feature of the data, simply because that patient reappears in the training data. Thus, keeping patient IDs lead to information leakage, since it reveals the data source. I would also remove the current length of hospital stay, since for a new patient this information is available only after the prediction is made.

**(ii)** A totally random assignment can be misleading here because we could be splitting a patient's admissions into two. Say patient 4 was admitted a total of 4 times. If we get a split where 2 admissions are in the training and 2 are in the testing split, then the model will have already seen the patient's information, leading to lower errors than they should be. Additionally, we are not guaranteed to have the admissions ordered; we could have later admissions in the testing split, with admissions that already happened in the training split. This is why it is important to keep patients whole (i.e., a patient's admssions are only in the training or the testing split). 

### Q1(b)

**(i)** This is a problem because we are standardizing on all the data; this means that we are taking into account the variation of both the training and testing sets, which will lead to artificially lower errors. What we want to do here is only standardize with the training data, within each CV fold. Similarly, using predictors with higher correlation to the outcome variable after training with the whole dataset is double-dipping. The data we are using is no longer independent of the model we are trying to fit, which will lead to artificial results. Predictor selection should be done before we actually train the model, using EDA.

**(ii)** Here, the $k$ chosen is valid; we generally want to choose the value that gets us the lowest CV validation error (or within one standard error). The mistake here is how we are reporting the model benchmark: using the lowest CV error is wrong, since we are optimizing the model with information that is already known to it. To fix this, we need to use the model with the chosen $k$, refit it on all of the training set (without the CV folds), and then use the testing set to report the error as the final evaluation.

### Q1(c)

**(i)** Write $p = p(x)$ and $\hat p = \hat p_D(x)$.

$$
\begin{aligned}
\mathbb{E}_{D,Y}\big[(Y-\hat p_D(x))^2\big] &= \mathbb{E}_{D,Y}\big[\big((Y-p)+(p-\hat p)\big)^2\big]\\
&= \mathbb{E}_{D,Y}\big[(Y-p)^2 + 2(Y-p)(p-\hat p) + (p-\hat p)^2\big]\\
&= \mathbb{E}_{D,Y}\big[(Y-p)^2\big] - 2\,\mathbb{E}_{D,Y}\big[(Y-p)(p-\hat p)\big] + \mathbb{E}_{D,Y}\big[(p-\hat p)^2\big]
\end{aligned}
$$

First term: $\mathbb{E}_{D,Y}[(Y-p)^2] = \operatorname{Var}(Y) = p(x)(1-p(x))$.

Cross term: since $Y \perp \hat p_D(x)$, $2\,\mathbb{E}[(Y-p)(p-\hat p)] = 2\,\mathbb{E}[Y-p]\,\mathbb{E}[p-\hat p]$, and $\mathbb{E}[Y]-p = p-p = 0$, so the cross term vanishes.

With $\mu(x) = \mathbb{E}_D[\hat p_D(x)]$:

$$
\begin{aligned}
\therefore\ \mathbb{E}_{D,Y}\big[(Y-\hat p_D(x))\big] &= \mathbb{E}_{D,Y}\big[(p(x)-\hat p_D(x))^2\big] + p(x)(1-p(x))\\
&= \mathbb{E}_{D,Y}\big[\big((p(x)+\mu(x)) + (\mu(x)-\hat p_D(x))\big)^2\big] + p(x)(1-p(x))
\end{aligned}
$$

where

$$
\begin{aligned}
\mathbb{E}_{D,Y}\big[\big((p(x)+\mu(x)) + (\mu(x)-\hat p_D(x))\big)^2\big]
&= \mathbb{E}\,(p(x)-\mu(x))^2 + 2\,\mathbb{E}_{D,Y}\big[(p(x)-\mu(x))(\mu(x)-\hat p_D(x))\big] + \mathbb{E}_{D,Y}\big[(\mu(x)-\hat p_D(x))^2\big]\\
&= \mathbb{E}\,(p(x)-\mu(x))^2 + 2\,(p(x)-\mu(x))\big(\mu(x)-\mathbb{E}[\hat p_D(x)]\big) + \mathbb{E}_{D,Y}\big[(\mu(x)-\hat p_D(x))^2\big]
\end{aligned}
$$

Since $\mathbb{E}[\hat p_D(x)] = \mu(x)$, the middle term is $0$, and $\mathbb{E}_{D,Y}[(\mu(x)-\hat p_D(x))^2] = \operatorname{Var}_D(\hat p_D(x))$, and ${E}\,(p(x)-\mu(x))^2 = (\mu(x) - p(x))^2$, since they're constant. Therefore

$$
\begin{aligned}
\mathbb{E}_{D,Y} &= \mathbb{E}\big[(p(x)-\mu(x))^2\big] + \mathbb{E}_{D,Y}\big[(\mu(x)-\hat p_D(x))^2\big] + p(x)(1-p(x))\\
&= (\mu(x)-p(x))^2 + \operatorname{Var}_D(\hat p_D(x)) + p(x)(1-p(x)).
\end{aligned}
$$

**(ii)** No. We always have the variance of $Y$, $p(x)(1-p(x))$, which is why, even if we fit perfectly, the variance from the random variable (our irreducible error) does not translate to perfect predictions.

## Q2: Regularization and shrinkage

### Q2(a)

**(i)** Let $L(\beta) = \|y - X\beta\|_2^2$ and
$P(\beta) = \|\beta\|_2^2 = \sum_j \beta_j^2$, or $\|\beta\|_1 = \sum_j |\beta_j|$, subject to $\|\beta\|_2^2 \le t$ or $\|\beta\|_1 \le t$, where $t = P(\hat\beta(\lambda))$,
and $P(\hat\beta) \le P(\hat\beta(\lambda)) \Rightarrow$ feasible.

$$L(\hat\beta) + \lambda P(\hat\beta) \le L(\beta) + \lambda P(\beta) \quad \forall \beta \in \mathbb{R}^p$$

For both ridge and lasso, $\|\beta\|_2^2 \le t$, $\|\beta\|_1 \le t$.

Since we are dealing with a minimizer,

$$
\begin{aligned}
L(\hat\beta) + \lambda P(\hat\beta) &\le L(\beta) + \lambda P(\beta)\\
L(\hat\beta) &\le L(\beta) + \lambda\big(P(\beta) - P(\hat\beta)\big) \qquad (\hat\beta \text{ is feasible})
\end{aligned}
$$

Assuming a feasible $\beta$, $P(\beta) \le P(\hat\beta)$, and $\lambda > 0$,

$$
\begin{aligned}
\therefore\ \lambda\big(P(\beta) - P(\hat\beta)\big) &\le 0\\
\Rightarrow\ L(\hat\beta) \le L(\beta) + \lambda\big(P(\beta) - P(\hat\beta)\big) &\le L(\beta) + 0\\
\Rightarrow\ L(\hat\beta) &\le L(\beta)
\end{aligned}
$$

Thus $\hat\beta$ is feasible, and $L(\hat\beta) \le L(\beta)$ for every feasible $\beta$, so $\hat\beta$ minimizes $L$ subject to $P(\beta) \le P(\hat\beta(\lambda))$.

**(ii)** We need to show that, for all $\beta \in \mathbb{R}^p$,

$$L(\tilde\beta) + \lambda P(\tilde\beta) \le L(\beta) + \lambda P(\beta).$$

*Proof.* $\tilde\beta$ is a constrained minimizer, which minimizes $L$ among feasible points, and $\hat\beta$ is feasible. Therefore,

$$L(\tilde\beta) \le L(\hat\beta).$$

Since $\tilde\beta$ minimizes $L$, $P(\tilde\beta) \le P(\hat\beta)$, and since $\lambda > 0$, multiplying by $\lambda$ keeps the inequality the same (it does not reverse it), so

$$\lambda P(\tilde\beta) \le \lambda P(\hat\beta).$$

Thus,

$$L(\tilde\beta) + \lambda P(\tilde\beta) \le L(\hat\beta) + \lambda P(\hat\beta).$$

From part (i), we showed that, for all $\beta \in \mathbb{R}^p$,

$$L(\hat\beta) + \lambda P(\hat\beta) \le L(\beta) + \lambda P(\beta).$$

Therefore, for all $\beta \in \mathbb{R}^p$,

$$L(\tilde\beta) + \lambda P(\tilde\beta) \le L(\hat\beta) + \lambda P(\hat\beta) \le L(\beta) + \lambda P(\beta). \qquad \blacksquare$$

### Q2(b)

Let $\hat\beta_1 = \hat\beta(\lambda_1)$, $\hat\beta_2 = \hat\beta(\lambda_2)$, $L_1 = L(\hat\beta_1)$, $L_2 = L(\hat\beta_2)$, $P_1 = P(\hat\beta_1)$, and $P_2 = P(\hat\beta_2)$.

Then, since each $\hat\beta_k$ is a minimizer, the following inequalities hold for every $\beta \in \mathbb{R}^p$:

$$
\begin{aligned}
L_1 + \lambda_1 P_1 &\le L(\beta) + \lambda_1 P(\beta). \quad \text{Choose } \beta = \hat\beta_2 \;\Rightarrow\; L_1 + \lambda_1 P_1 \le L_2 + \lambda_1 P_2\\
L_2 + \lambda_2 P_2 &\le L(\beta) + \lambda_2 P(\beta). \quad \text{Choose } \beta = \hat\beta_1 \;\Rightarrow\; L_2 + \lambda_2 P_2 \le L_1 + \lambda_2 P_1
\end{aligned}
$$

Adding the two inequalities and cancelling $L_1$ and $L_2$, which appear on both sides:

$$
\begin{aligned}
L_1 + \lambda_1 P_1 + L_2 + \lambda_2 P_2 &\le L_2 + \lambda_1 P_2 + L_1 + \lambda_2 P_1\\
\lambda_1 P_1 + \lambda_2 P_2 &\le \lambda_1 P_2 + \lambda_2 P_1\\
(\lambda_2 - \lambda_1) P_2 &\le (\lambda_2 - \lambda_1) P_1\\
(\lambda_2 - \lambda_1)\, P(\hat\beta(\lambda_2)) &\le (\lambda_2 - \lambda_1)\, P(\hat\beta(\lambda_1))
\end{aligned}
$$

Simplifying by dividing both sides by $(\lambda_2 - \lambda_1)$: since $\lambda_2 - \lambda_1 > 0$, this keeps the inequality the same (it does not reverse it).

$$\therefore\; P(\hat\beta(\lambda_2)) \le P(\hat\beta(\lambda_1)). \qquad \blacksquare$$

### Q2(c)

**(i)** _TODO_

**(ii)** _TODO_

### Q2(d)

**(i)** _TODO_

**(ii)** _TODO_

## Q3: CV and the one-standard-error rule

### Q3(a)

**(i)** _TODO_

**(ii)** _TODO_

### Q3(b)

**(i)** _TODO_

**(ii)** _TODO_

## Q4: Classification, margins, and decisions

### Q4(a)

**(i)** _TODO_

**(ii)** _TODO_

### Q4(b)

**(i)** _TODO_

**(ii)** _TODO_

### Q4 SVM options

**Chosen options:** _TODO_ and _TODO_ (exactly two of A, B, C, D)

### Q4 Option _TODO_

**(i)** _TODO_

**(ii)** _TODO_

### Q4 Option _TODO_

**(i)** _TODO_

**(ii)** _TODO_

## Q5: Trees, averaging, and regularized corrections

### Q5(a)

_TODO_

### Q5(b)

**(i)** _TODO_

**(ii)** _TODO_

### Q5(c)

**(i)** _TODO_

**(ii)** _TODO_

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "sklearn:", sklearn.__version__)
DATA = Path(".")
SEED = 2112

## Q6: Ridge implementation and prediction error
Keep the design fixed. The intercept is known to be zero. Use the Lecture 2 and Q2 objective: residual sum of squares plus lambda times the squared coefficient norm. Do not fit an intercept or recenter each response. The revised lambda labels 0, 2, 20, 200 give the same fits as 0, 1, 10, 100 under the earlier half-SSE convention.

In [ ]:
X_sim = pd.read_csv(DATA / "ridge_design.csv").to_numpy()
beta = np.array([1., 1., 0., 0., 0.])
x0 = np.array([1., 0., 0., 0., 0.])
lambdas = [0., 2., 20., 200.]
R = 500
assert X_sim.shape == (60, 5)
assert np.allclose(X_sim.mean(axis=0), 0)
rng = np.random.default_rng(SEED)
Y_sim = (X_sim @ beta)[None, :] + rng.normal(size=(R, len(X_sim)))
Y0 = x0 @ beta + rng.normal(size=R)

In [ ]:
def ridge_fit(X, y, lam):
    """Return ridge coefficients using a linear system solve. Q6(a)."""
    raise NotImplementedError("Implement Q6(a)")

# TODO: check (X.T @ X + lam*I) @ beta_hat - X.T @ y is close to zero.
# Use the first simulated response; report the largest absolute entry for each lambda.
# np.linalg.solve(A, v) solves A @ b = v without forming an inverse.

In [ ]:
# TODO Q6(b): reuse Y_sim for every lambda; fit, tabulate, and plot.
# Use np.var(predictions, ddof=0) for the finite-sample decomposition.
# MSE_f compares predictions with the true mean f0=1.
# Report bias, squared bias, variance and MSE_f; plot the last three.
# The retained Y0 variable is unused; no noisy-outcome comparison is required.

### Q6 interpretation
Replace this with your own explanation of the observed bias and variance tradeoff.

## Q7: Hitters — freeze choices before testing
Salary and log_salary are outcomes, not predictors. row_id only identifies rows. Do not use test outcomes in exploratory analysis, tuning, or preprocessing.

In [ ]:
data = pd.read_csv(DATA / "hitters_hw1.csv")
features = ["Years", "Hits", "AtBat", "HmRun", "Runs", "RBI", "Walks", "PutOuts"]
development = data.loc[data["split"] == "development"].copy()
test_rows = data.loc[data["split"] == "test"].copy()
X_dev = development[features]
y_dev = development["log_salary"]
fold_ids = development["fold"].to_numpy()
cv_splits = [(np.flatnonzero(fold_ids != k), np.flatnonzero(fold_ids == k)) for k in range(5)]
assert len(development) == 210 and len(test_rows) == 53
assert set(development.row_id).isdisjoint(test_rows.row_id)
assert all(len(va) == 42 for tr, va in cv_splits)

In [ ]:
# TODO Q7(a): report development/test and validation-fold counts; discuss a future season.
# TODO Q7(b): define the specified candidates and fit on the common folds.
# Ridge scaling must be inside a Pipeline fitted on each training fold.
# Include a training-fold mean predictor as the baseline.
# Compute mean validation MSE for each candidate; break ties by listed order.

### Computing reference (not additional theory)
`make_pipeline(StandardScaler(), Ridge(...))` combines standardization and ridge: fit the combined object only on the current training fold, then predict its validation fold.

| Method | Class and parameter names |
|---|---|
| Ridge | `Ridge(alpha=...)`; use `StandardScaler` with `make_pipeline` |
| Tree | `DecisionTreeRegressor(max_leaf_nodes=..., min_samples_leaf=5, random_state=SEED)` |
| Forest / bagging | `RandomForestRegressor(n_estimators=200, max_features=..., min_samples_leaf=5, bootstrap=True, max_depth=None, random_state=SEED)` |
| Boosting | `GradientBoostingRegressor(loss="squared_error", learning_rate=..., n_estimators=..., max_leaf_nodes=3, min_samples_leaf=5, random_state=SEED)` |

Use the candidate values in the PDF. `clone(model)` makes an unfitted copy for a new fold. The values in `cv_splits` are row positions; use `.iloc` for pandas rows. A residual is observed minus predicted log salary.

### Freeze the analysis
Record each family's selected settings and CV MSE, and your overall CV-selected predictor, before evaluating test predictions. A table is sufficient. Do not revise choices after seeing the test outcomes.

In [ ]:
# TODO Q7(c): refit each selected family on all development rows.
# Now evaluate all four selected models and the mean baseline once.
# Tabulate CV and test MSE; keep your preselected model unchanged.

In [ ]:
# TODO Q7(d): plot test residuals against predictions for the preselected model; add a zero line.

### Plot commentary (at most 150 words)
Describe one visible pattern and its implication, or explain that there is no clear pattern. Two to four sentences are sufficient.

## Acknowledgments
List collaborators discussed with and outside resources used. Follow the syllabus AI rules; the submitted work must be your own.